# 6. Multi-Seed Sweep — one cell per run (12 total)

Run setup cells 1–3 first, then run cells 4–15 in order.

**GPT** identity / W&B id: `tinystories_large_{arch}_ctx512_steps3000_seed{SEED}`
**VLM** run name: `vlm_{arch}_flickr30k_b8_seed{SEED}`

Seed `42` is excluded (already finished). Each run cell skips automatically if outputs already exist.

## 1. Sync repo and install deps

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-next-scale.git'


def is_repo_root(path: Path) -> bool:
    return (path / 'src' / 'training' / 'train.py').exists() and (path / 'requirements.txt').exists()


def discover_repo() -> Path | None:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AttnResGPT-next-scale')]
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate.resolve()
    for root in [Path('/content'), Path('/content/drive/MyDrive')]:
        if not root.exists():
            continue
        for path in root.rglob('AttnResGPT-next-scale'):
            if is_repo_root(path):
                return path.resolve()
    return None


REPO_DIR = discover_repo()
if REPO_DIR is None:
    target = Path('/content/AttnResGPT-next-scale')
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    REPO_DIR = target.resolve()

os.chdir(REPO_DIR)
print(f'Using repo at: {REPO_DIR}')
subprocess.run(['git', 'fetch', '--quiet'], check=False)
subprocess.run(['git', 'pull', '--ff-only'], check=False)
subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 2. W&B + GPU

In [ ]:
import os
import torch

# Optional: os.environ['WANDB_API_KEY'] = '<your-key>'
print('WANDB_API_KEY set =', bool(os.environ.get('WANDB_API_KEY')))
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Switch Colab runtime to GPU before launching sweeps.')

## 3. Launch helpers (used by run cells below)

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

SKIP_EXISTING = True
GPT_CONFIG = 'configs/large_ctx512_3000.yaml'
OUTPUT_ROOT = Path('outputs')  # logging.output_root in configs/large_ctx512_3000.yaml
VLM_BATCH_SIZE = 8
VLM_NUM_WORKERS = 2
VLM_FINAL_CHECKPOINT = 'step_0006750.pt'


def repo_root() -> Path:
    """Paths are relative to repo root (cell 1 runs os.chdir(REPO_DIR))."""
    root = Path.cwd().resolve()
    if (root / 'src' / 'training' / 'train.py').exists():
        return root
    fallback = Path('/content/AttnResGPT-next-scale').resolve()
    if (fallback / 'src' / 'training' / 'train.py').exists():
        return fallback
    raise RuntimeError('Run cell 1 first — cannot find repo root.')


def gpt_identity(architecture: str, seed: int) -> str:
    return f'tinystories_large_{architecture}_ctx512_steps3000_seed{seed}'


def gpt_paths(architecture: str, seed: int, *, root: Path | None = None) -> dict:
    root = root or repo_root()
    identity = gpt_identity(architecture, seed)
    run_dir = root / OUTPUT_ROOT / 'runs' / identity
    return {
        'root': root,
        'identity': identity,
        'run_dir': run_dir,
        'checkpoint_dir': root / OUTPUT_ROOT / 'checkpoints' / identity,
        'summary': run_dir / 'run_summary.json',
        'blockers': [
            run_dir / 'train_metrics.jsonl',
            run_dir / 'val_metrics.jsonl',
            run_dir / 'run_summary.json',
        ],
        'global_logs': sorted((root / OUTPUT_ROOT / 'logs').glob(f'{identity}_*.jsonl')),
    }


def reset_gpt_run(architecture: str, seed: int) -> None:
    """Delete partial/failed run outputs under outputs/ (same dirs train.py uses)."""
    paths = gpt_paths(architecture, seed)
    print(f'repo root: {paths["root"]}')
    print(f'resetting: {paths["identity"]}')
    for label in ('run_dir', 'checkpoint_dir'):
        path = paths[label]
        if path.exists():
            shutil.rmtree(path)
            print(f'  removed {label}: {path}')
    for path in paths['global_logs']:
        path.unlink(missing_ok=True)
        print(f'  removed log: {path}')
    remaining = [p for p in paths['blockers'] if p.exists()]
    if remaining:
        raise RuntimeError(f'cleanup incomplete: {remaining}')
    print('  ok — safe to launch')


def launch_gpt(architecture: str, seed: int) -> None:
    paths = gpt_paths(architecture, seed)
    identity = paths['identity']
    root = paths['root']
    if SKIP_EXISTING and paths['summary'].exists():
        print(f'SKIP (complete): {identity}')
        return
    if any(p.exists() for p in paths['blockers']):
        print(f'partial outputs for {identity} — clearing before launch')
        reset_gpt_run(architecture, seed)
    cmd = [
        sys.executable, '-m', 'src.training.train',
        '--config', GPT_CONFIG,
        '--model', architecture,
        '--overrides',
        f'experiment.seed={seed}',
        f'logging.wandb.tags=[multiseed,large,ctx512,{architecture},seed{seed}]',
    ]
    print(f'Launching {identity} from {root}')
    subprocess.run(cmd, check=True, cwd=root)


def vlm_run_name(architecture: str, seed: int) -> str:
    return f'vlm_{architecture}_flickr30k_b{VLM_BATCH_SIZE}_seed{seed}'


def launch_vlm(architecture: str, seed: int) -> None:
    root = repo_root()
    run_name = vlm_run_name(architecture, seed)
    ckpt = root / 'checkpoints' / run_name / VLM_FINAL_CHECKPOINT
    if SKIP_EXISTING and ckpt.exists():
        print(f'SKIP (exists): {run_name}')
        return
    cmd = [
        sys.executable, 'experiments/vlm_attnres_flickr30k.py',
        '--decoder-architecture', architecture,
        '--seed', str(seed),
        '--batch-size', str(VLM_BATCH_SIZE),
        '--num-workers', str(VLM_NUM_WORKERS),
        '--run-name', run_name,
    ]
    print(f'Launching {run_name} from {root}')
    subprocess.run(cmd, check=True, cwd=root)

### Run 1/12 — GPT baseline, seed 123
`tinystories_large_baseline_ctx512_steps3000_seed123`

In [ ]:
launch_gpt('baseline', 123)

### Run 2/12 — GPT attnres, seed 123
`tinystories_large_attnres_ctx512_steps3000_seed123`

In [ ]:
launch_gpt('attnres', 123)

### Run 3/12 — GPT baseline, seed 456
`tinystories_large_baseline_ctx512_steps3000_seed456`

In [ ]:
launch_gpt('baseline', 456)

### Run 4/12 — GPT attnres, seed 456
`tinystories_large_attnres_ctx512_steps3000_seed456`

In [ ]:
launch_gpt('attnres', 456)

### Run 5/12 — GPT baseline, seed 789
`tinystories_large_baseline_ctx512_steps3000_seed789`

In [ ]:
launch_gpt('baseline', 789)

### Run 6/12 — GPT attnres, seed 789
`tinystories_large_attnres_ctx512_steps3000_seed789`

In [ ]:
launch_gpt('attnres', 789)

### Run 7/12 — VLM baseline, seed 123
`vlm_baseline_flickr30k_b8_seed123`

In [ ]:
launch_vlm('baseline', 123)

### Run 8/12 — VLM attnres, seed 123
`vlm_attnres_flickr30k_b8_seed123`

In [ ]:
launch_vlm('attnres', 123)

### Run 9/12 — VLM baseline, seed 456
`vlm_baseline_flickr30k_b8_seed456`

In [ ]:
launch_vlm('baseline', 456)

### Run 10/12 — VLM attnres, seed 456
`vlm_attnres_flickr30k_b8_seed456`

In [ ]:
launch_vlm('attnres', 456)

### Run 11/12 — VLM baseline, seed 789
`vlm_baseline_flickr30k_b8_seed789`

In [ ]:
launch_vlm('baseline', 789)

### Run 12/12 — VLM attnres, seed 789
`vlm_attnres_flickr30k_b8_seed789`

In [ ]:
launch_vlm('attnres', 789)

## Post-sweep sanity check

In [ ]:
import json
from pathlib import Path

SEEDS = [123, 456, 789]
ARCHITECTURES = ['baseline', 'attnres']

print('=== GPT summaries ===')
for architecture in ARCHITECTURES:
    for seed in SEEDS:
        identity = gpt_identity(architecture, seed)
        summary_path = Path('outputs/runs') / identity / 'run_summary.json'
        if not summary_path.exists():
            print(f'MISSING {identity}')
            continue
        summary = json.loads(summary_path.read_text())
        print(
            f"{identity}: val_loss={summary.get('val_loss'):.4f} "
            f"ppl={summary.get('perplexity'):.2f}"
        )

print('\n=== VLM checkpoints ===')
for architecture in ARCHITECTURES:
    for seed in SEEDS:
        run_name = vlm_run_name(architecture, seed)
        ckpt = Path('checkpoints') / run_name / VLM_FINAL_CHECKPOINT
        print(f"{run_name}: {'OK' if ckpt.exists() else 'MISSING'}")